# Data Agent Evaluation Notebook

This notebook evaluates a Fabric Data Agent's accuracy by:
1. Executing DAX queries against the semantic model to generate **ground truth**
2. Querying the Data Agent via the **Fabric Data Agent SDK**
3. Using an **LLM-as-judge** (gpt-5) to compare agent responses to ground truth
4. **Calibrating the judge** to measure its own reliability
5. Producing a **calibration-adjusted final score**

In [1]:
%pip install -U fabric-data-agent-sdk openai --q

Reason for being yanked: Yanked due to conflicts with CVE-2024-35195 mitigation
Note: you may need to restart the kernel to use updated packages.


## Step 1: Imports & Configuration

In [2]:
import sempy.fabric as fabric
import pandas as pd
import time

workspace_id = "3261e461-10ce-477e-9b93-824bb2b425b9"
semantic_model_id = "e6b14de2-055f-4dc1-aff5-27f49936b37e"
data_agent_name = "mcaps_eval_dataagent"  # Display name of the existing agent

## Step 2: Define DAX Queries (Ground Truth)

In [3]:
dax1 = """
// Top 3 customers in Sao Paulo based on number of approved orders
DEFINE
  // Filter table for customers in Sao Paulo
  VAR _SaoPauloCustomers =
    FILTER(
      ALL('customers'),
      'customers'[customer_city] = "Sao Paulo"
    )

  // Summary table: approved order count per customer in Sao Paulo
  VAR _SummaryTable =
    SUMMARIZECOLUMNS(
      'customers'[customer_id],
      'customers'[customer_city],
      _SaoPauloCustomers, // apply city filter as table argument
      "Approved Orders",
        CALCULATE(
          COUNTROWS('orders'),
          'orders'[order_status] = "approved"
        )
    )

// Return top 3 customers by approved order count in Sao Paulo
EVALUATE
  TOPN(
    3,
    _SummaryTable,
    [Approved Orders],
    DESC,
    'customers'[customer_id],
    ASC
  )
ORDER BY
  [Approved Orders] DESC,
  'customers'[customer_id] ASC
"""

dax2="""
// City with the highest number of customers whose orders were not fulfilled
DEFINE
  VAR _UnfulfilledByCity =
    SUMMARIZECOLUMNS(
      'customers'[customer_city],
      // Filter to orders that were not delivered to the customer
      FILTER(
        ALL('orders'),
        ISBLANK('orders'[order_delivered_customer_date])
      ),
      // Count distinct customers with at least one unfulfilled order in each city
      "Unfulfilled Customers", DISTINCTCOUNT('orders'[customer_id])
    )

EVALUATE
  TOPN(
    1,
    _UnfulfilledByCity,
    [Unfulfilled Customers],
    DESC
  )
ORDER BY
  [Unfulfilled Customers] DESC,
  'customers'[customer_city] ASC
  """

questions_dax = [
    {
        "question": "Who are the top 3 customers in Sao Paulo based on number of approved orders?",
        "dax": dax1
    },
    {
        "question": "Which city had the most customers with unfulfilled orders?",
        "dax": dax2
    }
]

df = pd.DataFrame(questions_dax)
display(df)

## Step 3: Refresh Semantic Model & Execute DAX Ground Truth

In [4]:
# Refresh the semantic model to ensure data is up to date
refresh_model = fabric.refresh_dataset(semantic_model_id, workspace_id)
time.sleep(10)
fabric.get_refresh_execution_details(semantic_model_id, refresh_model, workspace_id)

RefreshExecutionDetails(start_time=datetime.datetime(2026, 4, 7, 23, 40, 26, 713000, tzinfo=datetime.timezone.utc), end_time=datetime.datetime(2026, 4, 7, 23, 40, 31, 883000, tzinfo=datetime.timezone.utc), type='Automatic', commit_mode='Transactional', status='Completed', extended_status='Completed', current_refresh_type='Automatic', number_of_attempts=1, objects=       Table  Partition     Status
0  customers  customers  Completed
1   dim_date   dim_date  Completed
2     orders     orders  Completed, messages=Empty DataFrame
Columns: [Message, Type]
Index: [], refresh_attempts=   Attempt Id                       Start Time  \
0           1 2026-04-07 23:40:27.459107+00:00   

                          End Time  Type  
0 2026-04-07 23:40:31.884691+00:00  Data  )

In [5]:
def get_dax_responses(df, semantic_model_id, workspace_id):
    """Execute each DAX query against the semantic model and store results."""
    responses = [
        fabric.evaluate_dax(semantic_model_id, dax, workspace_id)
        for dax in df["dax"]
    ]
    df = df.copy()
    df["dax_response"] = responses
    return df

df_with_responses = get_dax_responses(df, semantic_model_id, workspace_id)
display(df_with_responses)

## Step 4: Format DAX Results as Expected Answers

Convert raw DAX DataFrame results into human-readable text strings
that the LLM judge can compare against the Data Agent's responses.

In [6]:
def format_dax_result_as_text(df_result):
    """Convert a DAX result DataFrame into a human-readable expected answer string."""
    if df_result is None or df_result.empty:
        return "No results"

    # Single-value result
    if len(df_result) == 1 and len(df_result.columns) == 1:
        return str(df_result.iloc[0, 0])

    # Single-row, multi-column result
    if len(df_result) == 1:
        parts = [f"{col}: {df_result.iloc[0][col]}" for col in df_result.columns]
        return ", ".join(parts)

    # Multi-row result: format as numbered list
    lines = []
    for idx, (_, row) in enumerate(df_result.iterrows(), 1):
        parts = [f"{col}: {row[col]}" for col in df_result.columns]
        lines.append(f"{idx}. " + ", ".join(parts))
    return "\n".join(lines)


# Build expected answers from DAX results
expected_answers = [
    format_dax_result_as_text(resp) for resp in df_with_responses["dax_response"]
]

# Create evaluation DataFrame
eval_df = pd.DataFrame({
    "question": df_with_responses["question"].tolist(),
    "expected_answer": expected_answers
})

print("Evaluation DataFrame:")
display(eval_df)
for i, row in eval_df.iterrows():
    print(f"\nQ{i+1}: {row['question']}")
    print(f"Expected: {row['expected_answer']}")

Evaluation DataFrame:



Q1: Who are the top 3 customers in Sao Paulo based on number of approved orders?
Expected: customers[customer_id]: b2191912d8ad6eac2e4dc3b6e1459515, customers[customer_city]: sao paulo, [Approved Orders]: 1

Q2: Which city had the most customers with unfulfilled orders?
Expected: customers[customer_city]: sao paulo, [Unfulfilled Customers]: 494


## Step 5: Connect to Data Agent

In [7]:
from fabric.dataagent.client import FabricDataAgentManagement

data_agent = FabricDataAgentManagement(data_agent_name, workspace=workspace_id)
print(f"Connected to Data Agent: {data_agent_name}")

Connected to Data Agent: mcaps_eval_dataagent


## Step 6: Define LLM-as-Judge Critic Prompt

Two prompts are defined here that must stay aligned:

- `critic_prompt` — passed to `evaluate_data_agent()`. The SDK injects the actual agent answer as a prior conversation turn, so this template only needs `{query}` and `{expected_answer}`.
- `calibration_judge_prompt` — used by the calibration harness, which runs outside the SDK. It must evaluate by the same rules as `critic_prompt` but receives `{actual_answer}` explicitly as a template variable.

Keeping the evaluation criteria identical between both prompts is what makes the calibration score meaningful.

In [8]:
critic_prompt = """You are an impartial judge evaluating whether a data agent's answer is correct, using the ground truth as the reference.

Rules (apply in order):
1. Numerical accuracy: values must match within 1% relative tolerance. Example: 1,234,567 vs 1,234,000 is YES; 1,234,567 vs 2,500,000 is NO.
2. Entity completeness: every entity in the ground truth (customer IDs, city names, category names) must appear in the answer. A missing entity is NO even if everything else matches.
3. Rank order: for ranked or ordered results, rank positions must match. Correct entities in the wrong rank is NO.
4. Semantic equivalence: ignore formatting differences (table vs list, markdown vs plain text, currency symbols, capitalisation). Treat accented and unaccented versions of the same name as equivalent (e.g. Sao Paulo = São Paulo).
5. Statistical measures: mean, median, mode, sum, and count are not interchangeable. A wrong measure is NO even if the value matches.
6. Extra information: an answer that contains all ground truth data plus additional non-contradictory context is still YES.
7. Refusals: if the answer says it cannot find data but the ground truth contains data, that is NO.
8. Time periods: if the answer uses a different time period than the ground truth, that is NO.

Respond with exactly one word: Yes or No.

Query: {query}

Ground Truth: {expected_answer}
"""

print("critic_prompt defined.")
print(f"Length: {len(critic_prompt)} characters")

critic_prompt defined.
Length: 1360 characters


In [9]:
# calibration_judge_prompt must use the same evaluation criteria as critic_prompt.
# The only difference is that it receives {actual_answer} explicitly as a
# template variable, because it runs outside the SDK evaluation loop.

calibration_judge_prompt = """You are an impartial judge evaluating whether a data agent's answer is correct, using the ground truth as the reference.

Rules (apply in order):
1. Numerical accuracy: values must match within 1% relative tolerance. Example: 1,234,567 vs 1,234,000 is YES; 1,234,567 vs 2,500,000 is NO.
2. Entity completeness: every entity in the ground truth (customer IDs, city names, category names) must appear in the answer. A missing entity is NO even if everything else matches.
3. Rank order: for ranked or ordered results, rank positions must match. Correct entities in the wrong rank is NO.
4. Semantic equivalence: ignore formatting differences (table vs list, markdown vs plain text, currency symbols, capitalisation). Treat accented and unaccented versions of the same name as equivalent (e.g. Sao Paulo = São Paulo).
5. Statistical measures: mean, median, mode, sum, and count are not interchangeable. A wrong measure is NO even if the value matches.
6. Extra information: an answer that contains all ground truth data plus additional non-contradictory context is still YES.
7. Refusals: if the answer says it cannot find data but the ground truth contains data, that is NO.
8. Time periods: if the answer uses a different time period than the ground truth, that is NO.

Respond with exactly one word: Yes or No.

Query: {query}

Ground Truth: {expected_answer}

Actual Answer: {actual_answer}
"""

print("calibration_judge_prompt defined.")

calibration_judge_prompt defined.


## Step 7: Calibrate the Judge (Meta-Evaluation)

Before running the actual evaluation, we test the critic prompt against
known-good and known-bad answer pairs to measure judge reliability.
This gives us precision, recall, F1 and error rates to adjust the final score.

In [10]:
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
import openai

judge_client = openai.AzureOpenAI(
    http_client=get_openai_httpx_sync_client(),
    api_version="2025-04-01-preview",
)

# ---------------------------------------------------------------------------
# Calibration cases — designed to hit the failure modes judges get wrong most.
# Each case documents WHY it is included so you can diagnose specific failures.
# ---------------------------------------------------------------------------
calibration_cases = [
    # ── TRUE POSITIVES (judge must say yes) ────────────────────────────────

    # Basic exact match in a sentence
    {
        "query": "How many approved orders are there?",
        "expected": "150",
        "actual": "There are 150 approved orders in the dataset.",
        "expected_verdict": "yes",
        "note": "Exact number embedded in prose"
    },
    # Accent normalisation
    {
        "query": "Which city has the most customers?",
        "expected": "Sao Paulo",
        "actual": "The city with the most customers is São Paulo.",
        "expected_verdict": "yes",
        "note": "Accent equivalence: Sao Paulo = São Paulo"
    },
    # Rounding within 1% tolerance
    {
        "query": "What is the total revenue?",
        "expected": "1,234,567.89",
        "actual": "Total revenue is $1,234,568.",
        "expected_verdict": "yes",
        "note": "Rounded to nearest dollar, within 1%"
    },
    # Table-to-prose, all entities and ranks preserved
    {
        "query": "Top 3 customers by order count?",
        "expected": "1. customer_id: C001, orders: 15\n2. customer_id: C002, orders: 12\n3. customer_id: C003, orders: 10",
        "actual": "The top 3 customers are C001 with 15 orders, C002 with 12 orders, and C003 with 10 orders.",
        "expected_verdict": "yes",
        "note": "Ranked list to prose, same rank order"
    },
    # Extra non-contradictory context
    {
        "query": "What is the average call handle time?",
        "expected": "4 minutes 32 seconds",
        "actual": "The average call handle time is 4 minutes and 32 seconds. This is above the industry benchmark of 3 minutes 45 seconds.",
        "expected_verdict": "yes",
        "note": "Extra benchmark context should not penalise"
    },
    # Compact number format
    {
        "query": "What is the monthly recurring revenue?",
        "expected": "Monthly recurring revenue is $2,400,000.",
        "actual": "MRR is $2.4M.",
        "expected_verdict": "yes",
        "note": "$2.4M == $2,400,000 within tolerance"
    },
    # Fraction vs percentage
    {
        "query": "What is the first call resolution rate?",
        "expected": "72%",
        "actual": "The first call resolution rate is 0.72.",
        "expected_verdict": "yes",
        "note": "Fraction vs percentage representation"
    },

    # ── TRUE NEGATIVES (judge must say no) ─────────────────────────────────

    # Wrong number, clearly outside tolerance
    {
        "query": "How many approved orders are there?",
        "expected": "150",
        "actual": "There are 250 approved orders.",
        "expected_verdict": "no",
        "note": "250 vs 150 — far outside 1%"
    },
    # Wrong entity
    {
        "query": "Which city has the most customers?",
        "expected": "Sao Paulo",
        "actual": "The city with the most customers is Rio de Janeiro.",
        "expected_verdict": "no",
        "note": "Wrong city entirely"
    },
    # Wrong magnitude
    {
        "query": "What is the total revenue?",
        "expected": "1,234,567",
        "actual": "Total revenue was R$2,500,000.",
        "expected_verdict": "no",
        "note": "Different number, different magnitude"
    },
    # One entity swapped in a ranked list
    {
        "query": "Top 3 customers by order count?",
        "expected": "1. customer_id: C001, orders: 15\n2. customer_id: C002, orders: 12\n3. customer_id: C003, orders: 10",
        "actual": "The top 3 customers are C001 with 15 orders, C004 with 12 orders, and C003 with 10 orders.",
        "expected_verdict": "no",
        "note": "C002 replaced by C004 — one wrong entity"
    },
    # Correct entities, wrong rank order
    {
        "query": "Top 3 product categories by complaint volume?",
        "expected": "1. Mobile plans\n2. Broadband\n3. Equipment",
        "actual": "Broadband (1st), Mobile plans (2nd), Equipment (3rd).",
        "expected_verdict": "no",
        "note": "All entities present but rank order inverted"
    },
    # Missing one entity from a list
    {
        "query": "List all cities with churn rate above 20%.",
        "expected": "North (24%), West (22%), Central (21%)",
        "actual": "North with 24% and West with 22% have churn rates above 20%.",
        "expected_verdict": "no",
        "note": "Central (21%) silently dropped"
    },
    # Agent refusal when data exists
    {
        "query": "How many calls were flagged for quality review in January?",
        "expected": "317 calls were flagged for quality review in January.",
        "actual": "I was unable to find data on quality review flags for January.",
        "expected_verdict": "no",
        "note": "Refusal when ground truth has a concrete answer"
    },
    # Wrong time period, same metric
    {
        "query": "What was the churn rate in Q3?",
        "expected": "Churn rate in Q3 was 14.2%.",
        "actual": "The churn rate was 14.2% in Q4.",
        "expected_verdict": "no",
        "note": "Correct value, wrong quarter — hardest for judges"
    },
    # Mean vs median — statistically distinct measures
    {
        "query": "What is the median call duration?",
        "expected": "6 minutes 10 seconds",
        "actual": "The average call duration is 6 minutes 10 seconds.",
        "expected_verdict": "no",
        "note": "Mean ≠ median even if value is identical"
    },
]

# ---------------------------------------------------------------------------
# Run calibration
# ---------------------------------------------------------------------------
judge_results = []
for case in calibration_cases:
    filled_prompt = calibration_judge_prompt.format(
        query=case["query"],
        expected_answer=case["expected"],
        actual_answer=case["actual"]
    )
    response = judge_client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": filled_prompt}],
        temperature=1,
    )
    raw_verdict = response.choices[0].message.content.strip().lower()

    # Normalise: strip punctuation, map variations to yes/no
    # The prompt asks for exactly "Yes" or "No" but defensively handle edge cases.
    if raw_verdict.startswith("yes"):
        judge_verdict = "yes"
    elif raw_verdict.startswith("no"):
        judge_verdict = "no"
    else:
        # Unexpected output — treat as wrong in both directions
        judge_verdict = f"unexpected: {raw_verdict}"

    is_correct = judge_verdict == case["expected_verdict"]
    judge_results.append({
        "query": case["query"][:50],
        "note": case.get("note", ""),
        "expected_verdict": case["expected_verdict"],
        "judge_verdict": judge_verdict,
        "correct": is_correct
    })

judge_cal_df = pd.DataFrame(judge_results)
display(judge_cal_df)

# ---------------------------------------------------------------------------
# Calibration metrics
# ---------------------------------------------------------------------------
tp = judge_cal_df[(judge_cal_df["expected_verdict"] == "yes") & (judge_cal_df["judge_verdict"] == "yes")].shape[0]
fp = judge_cal_df[(judge_cal_df["expected_verdict"] == "no")  & (judge_cal_df["judge_verdict"] == "yes")].shape[0]
fn = judge_cal_df[(judge_cal_df["expected_verdict"] == "yes") & (judge_cal_df["judge_verdict"] == "no")].shape[0]
tn = judge_cal_df[(judge_cal_df["expected_verdict"] == "no")  & (judge_cal_df["judge_verdict"] == "no")].shape[0]

total_pos = tp + fn
total_neg = tn + fp

judge_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
judge_recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
judge_f1        = 2 * judge_precision * judge_recall / (judge_precision + judge_recall) if (judge_precision + judge_recall) > 0 else 0
judge_accuracy  = judge_cal_df["correct"].sum() / len(judge_cal_df)
judge_tpr       = judge_recall
judge_fpr       = fp / total_neg if total_neg > 0 else 0

print(f"\n{'='*50}")
print(f"  JUDGE CALIBRATION RESULTS")
print(f"{'='*50}")
print(f"  Total test cases:   {len(judge_cal_df)}")
print(f"  Correct verdicts:   {judge_cal_df['correct'].sum()}/{len(judge_cal_df)} ({judge_accuracy*100:.0f}%)")
print(f"  Precision:          {judge_precision:.2f}")
print(f"  Recall:             {judge_recall:.2f}")
print(f"  F1 Score:           {judge_f1:.2f}")
print(f"  True Positive Rate: {judge_tpr:.2f}")
print(f"  False Positive Rate:{judge_fpr:.2f}")
print(f"{'='*50}")

# Surface individual failures to make debugging easier
failures = judge_cal_df[~judge_cal_df["correct"]]
if not failures.empty:
    print(f"\nFailed cases ({len(failures)}):")
    for _, row in failures.iterrows():
        print(f"  [{row['expected_verdict']} → {row['judge_verdict']}] {row['query'][:60]} | {row['note']}")

if judge_accuracy < 0.75:
    print("\nWARNING: Judge accuracy below 75%. Refine the critic prompt before proceeding.")
elif judge_accuracy < 0.90:
    print("\nJudge accuracy is moderate. Interpret results with the confidence range in Step 11.")
else:
    print("\nJudge accuracy is high. Critic prompt is reliable.")


  JUDGE CALIBRATION RESULTS
  Total test cases:   16
  Correct verdicts:   16/16 (100%)
  Precision:          1.00
  Recall:             1.00
  F1 Score:           1.00
  True Positive Rate: 1.00
  False Positive Rate:0.00

Judge accuracy is high. Critic prompt is reliable.


## Step 8: Run Data Agent Evaluation

Use the SDK's `evaluate_data_agent()` with the critic prompt to
query the agent, compare responses, and store results.

In [11]:
from fabric.dataagent.evaluation import evaluate_data_agent

table_name = "eval_results"
data_agent_stage = "production"

# workspace_name should be None if notebook runs in the same workspace as the agent,
# or the workspace display NAME (not ID) if in a different workspace.
evaluation_id = evaluate_data_agent(
    eval_df,
    data_agent_name,
    workspace_name=None,
    critic_prompt=critic_prompt,
    table_name=table_name,
    data_agent_stage=data_agent_stage
)

print(f"Evaluation ID: {evaluation_id}")

100%|██████████| 2/2 [00:35<00:00, 17.80s/it]


Evaluation ID: 17e5bfb4-7acf-45bf-9811-64e6d22302f4


## Step 9: View Evaluation Summary

In [12]:
from fabric.dataagent.evaluation import get_evaluation_summary

eval_summary = get_evaluation_summary(table_name)
display(eval_summary)

## Step 10: View Detailed Results

In [13]:
from fabric.dataagent.evaluation import get_evaluation_details

eval_details = get_evaluation_details(
    evaluation_id,
    table_name,
    get_all_rows=True,
    verbose=True
)
display(eval_details)

index,question,expected_answer,actual_answer,evaluation_judgement,thread_url
0,Which city had the most customers with unfulfilled orders?,"customers[customer_city]: sao paulo, [Unfulfilled Customers]: 494","The city with the most customers who have unfulfilled orders is São Paulo, with 495 customers.",True,thread_bjYAkbjfXfyj8LrQREWT0AAn
1,Who are the top 3 customers in Sao Paulo based on number of approved orders?,"customers[customerid]: b2191912d8ad6eac2e4dc3b6e1459515, customers[customercity]: sao paulo, [Approved Orders]: 1",The top customer in Sao Paulo based on the number of approved orders is: Customer ID: b2191912d8ad6eac2e4dc3b6e1459515 with 1 approved order. No other customers in Sao Paulo had more than 1 approved order in the available data.,True,thread_T238yx1z0ZpOMy9gXOsqtl1G


## Step 11: Calibration-Adjusted Final Score

The raw evaluation score assumes the judge is perfect. Since we calibrated the
judge and know its error rates (FPR, FNR), we compute a **debiased estimate**
of the true agent accuracy using:

```
adjusted_accuracy = (observed_pass_rate - FPR) / (TPR - FPR)
```

In [14]:
# Discover actual column names from the SDK
print("Columns returned by SDK:", eval_details.columns.tolist())

# The SDK may name the result column differently across versions.
# Try common variants: evaluation_result, eval_result, result, is_correct
result_col = None
for candidate in ["evaluation_result", "eval_result", "result", "is_correct", "pass"]:
    if candidate in eval_details.columns:
        result_col = candidate
        break

if result_col is None:
    # Fallback: look for any column containing 'result' or 'eval'
    for col in eval_details.columns:
        if "result" in col.lower() or "eval" in col.lower():
            result_col = col
            break

if result_col is None:
    raise RuntimeError(
        f"Cannot find evaluation result column. Available columns: {eval_details.columns.tolist()}"
    )

print(f"Using result column: '{result_col}'")
print(f"Unique values: {eval_details[result_col].unique()}")

# Count passed: handle both boolean True and string "Yes"/"True"
total = len(eval_details)
result_vals = eval_details[result_col]
if result_vals.dtype == bool:
    passed = result_vals.sum()
else:
    passed = result_vals.astype(str).str.strip().str.lower().isin(["true", "yes", "1"]).sum()
failed = total - passed
raw_accuracy = (passed / total) * 100 if total > 0 else 0

# Observed pass rate
observed_pass_rate = passed / total if total > 0 else 0

# Debiased estimate using judge calibration error rates
if (judge_tpr - judge_fpr) > 0:
    adjusted_pass_rate = (observed_pass_rate - judge_fpr) / (judge_tpr - judge_fpr)
    adjusted_pass_rate = max(0.0, min(1.0, adjusted_pass_rate))  # clamp to [0, 1]
else:
    adjusted_pass_rate = observed_pass_rate  # fallback if judge is random

# Confidence bounds based on judge accuracy
lower_bound = max(0, adjusted_pass_rate - (1 - judge_accuracy))
upper_bound = min(1, adjusted_pass_rate + (1 - judge_accuracy))

print(f"\n{'='*60}")
print(f"  FINAL EVALUATION REPORT")
print(f"{'='*60}")
print(f"  Total Questions:          {total}")
print(f"  Raw Passed (judge says):  {passed} ✅  /  {failed} ❌")
print(f"  Raw Accuracy:             {raw_accuracy:.1f}%")
print(f"{'─'*60}")
print(f"  Judge Calibration:")
print(f"    Judge Accuracy:         {judge_accuracy*100:.0f}%")
print(f"    Judge Precision:        {judge_precision:.2f}")
print(f"    Judge Recall:           {judge_recall:.2f}")
print(f"    Judge F1:               {judge_f1:.2f}")
print(f"    False Positive Rate:    {judge_fpr:.2f}")
print(f"    False Negative Rate:    {1-judge_tpr:.2f}")
print(f"{'─'*60}")
print(f"  Calibration-Adjusted Score:")
print(f"    Adjusted Accuracy:      {adjusted_pass_rate*100:.1f}%")
print(f"    Confidence Range:       [{lower_bound*100:.1f}% — {upper_bound*100:.1f}%]")
print(f"{'='*60}")

if judge_accuracy >= 0.9:
    print("\n✅ High judge reliability — adjusted score is trustworthy.")
elif judge_accuracy >= 0.75:
    print("\n⚡ Moderate judge reliability — interpret with caution, see confidence range.")
else:
    print("\n⚠️  Low judge reliability — raw score may be unreliable. Refine critic prompt.")

Columns returned by SDK: ['id', 'evaluation_id', 'thread_id', 'run_timestamp', 'question', 'expected_answer', 'actual_answer', 'execution_time_sec', 'status', 'thread_url', 'evaluation_judgement', 'evaluation_message', 'evaluation_status', 'evaluation_thread_url', 'data_agent_version', 'data_agent_etag', 'data_agent_last_updated', 'data_agent_configuration', 'data_sources']
Using result column: 'evaluation_id'
Unique values: ['17e5bfb4-7acf-45bf-9811-64e6d22302f4']

  FINAL EVALUATION REPORT
  Total Questions:          2
  Raw Passed (judge says):  0 ✅  /  2 ❌
  Raw Accuracy:             0.0%
────────────────────────────────────────────────────────────
  Judge Calibration:
    Judge Accuracy:         100%
    Judge Precision:        1.00
    Judge Recall:           1.00
    Judge F1:               1.00
    False Positive Rate:    0.00
    False Negative Rate:    0.00
────────────────────────────────────────────────────────────
  Calibration-Adjusted Score:
    Adjusted Accuracy:      0